# Data preparation
Load a project CSV, inspect label balance, normalize Unicode, and create deterministic train, validation, and test splits. Keep raw data unchanged.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = Path('../data')
CSV_PATH = DATA / 'raw' / 'your_dataset.csv'  # change this
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'sentiment'

df = pd.read_csv(CSV_PATH).dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.normalize('NFC').str.strip()
df = df[df[TEXT_COLUMN].ne('')]
print(df.shape)
display(df[LABEL_COLUMN].value_counts(dropna=False))
display(df.head())

In [ ]:
# Fixed seed makes the split repeatable. Stratification preserves class proportions when possible.
train, holdout = train_test_split(df, test_size=0.30, random_state=42, stratify=df[LABEL_COLUMN])
validation, test = train_test_split(holdout, test_size=0.50, random_state=42, stratify=holdout[LABEL_COLUMN])
for name, frame in [('train', train), ('validation', validation), ('test', test)]:
    destination = DATA / name
    destination.mkdir(parents=True, exist_ok=True)
    frame.to_csv(destination / 'dataset.csv', index=False)
    print(name, len(frame), frame[LABEL_COLUMN].value_counts().to_dict())